In [1]:
from pypdf import PdfWriter, PdfReader
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from datetime import datetime
import io
import os

def create_title_page(writer, pdf_files):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    
    # Add timestamp and PDF count header
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    c.setFont("Helvetica-Bold", 14)
    header = f"{len(pdf_files)} PDFs merged on {timestamp}"
    c.drawString(72, 750, header)
    
    # Add separator line
    c.line(72, 735, 540, 735)
    
    # List of articles
    c.setFont("Helvetica", 12)
    y_position = 700
    
    for i, pdf_path in enumerate(pdf_files, 1):
        pdf_basename = os.path.basename(pdf_path).split('.')[0]
        title = f"Article {i}: {pdf_basename}"
        c.drawString(72, y_position, title)
        y_position -= 20
        
        if y_position < 50:
            c.showPage()
            c.setFont("Helvetica", 12)
            y_position = 750
    
    c.showPage()
    c.save()
    packet.seek(0)
    title_pdf = PdfReader(packet)
    writer.add_page(title_pdf.pages[0])

def create_sample_pdf(filename, title, bookmarks=None):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    
    # Main content
    c.setFont("Helvetica", 12)
    c.drawString(100, 750, title)
    c.showPage()
    c.save()
    packet.seek(0)

    new_pdf = PdfReader(packet)
    writer = PdfWriter()
    writer.add_page(new_pdf.pages[0])

    # Add bookmarks only if provided
    if bookmarks:
        for bookmark_title, page in bookmarks:
            writer.add_outline_item(bookmark_title, page)

    with open(filename, 'wb') as f:
        writer.write(f)

def create_unstructured_pdf(filename, title):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    c.setFont("Helvetica", 12)
    c.drawString(100, 750, title)
    c.showPage()
    c.save()
    packet.seek(0)

    new_pdf = PdfReader(packet)
    writer = PdfWriter()
    writer.add_page(new_pdf.pages[0])

    with open(filename, 'wb') as f:
        writer.write(f)

def add_separator_page(writer, pdf_basename, full_path, metadata):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    
    c.setFont("Helvetica-Bold", 16)
    c.setFillColorRGB(0, 0, 1)  # Blue
    
    text = pdf_basename
    x, y = 100, 600
    c.linkURL(full_path, (x, y-5, x+200, y+20), relative=1)
    c.drawString(x, y, text)
    
    c.setFont("Helvetica", 12)
    c.setFillColorRGB(0, 0, 0)  # Black
    y_position = 500
    for key, value in metadata.items():
        c.drawString(100, y_position, f"{key}: {value}")
        y_position -= 30
    
    c.showPage()
    c.save()
    packet.seek(0)
    separator_pdf = PdfReader(packet)
    writer.add_page(separator_pdf.pages[0])

def add_margin_text(page, basename):
    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=letter)
    
    # Add vertical margin text
    c.saveState()
    c.setFont("Helvetica-Bold", 14)
    c.setFillColor(colors.darkgreen)
    c.translate(580, 400)
    c.rotate(90)
    c.drawString(0, 0, basename)
    c.restoreState()
    
    c.showPage()
    c.save()
    packet.seek(0)
    
    margin_pdf = PdfReader(packet)
    margin_page = margin_pdf.pages[0]
    margin_page.merge_page(page)
    return margin_page

def merge_pdfs_with_structure(pdf_files, output_path):
    writer = PdfWriter()
    
    # Add title page first
    create_title_page(writer, pdf_files)
    
    sample_metadata = [
        {
            "Title": "First Research Paper",
            "Author": "John Smith",
            "Source": "Science Journal",
            "Date": "2024-01-15"
        },
        {
            "Title": "Second Research Paper",
            "Author": "Jane Doe",
            "Source": "Nature",
            "Date": "2024-02-20"
        },
        {
            "Title": "Third Research Paper",
            "Author": "Bob Johnson",
            "Source": "Research Quarterly",
            "Date": "2024-03-10"
        }
    ]
    
    current_page = 1
    
    for i, pdf_path in enumerate(pdf_files, 1):
        pdf_basename = os.path.basename(pdf_path).split('.')[0]
        full_path = os.path.abspath(pdf_path)
        
        add_separator_page(writer, pdf_basename, full_path, sample_metadata[i-1])
        article_title = f"Article {i}: {pdf_basename}"
        separator_bookmark = writer.add_outline_item(article_title, current_page)
        current_page += 1
        
        pdf = PdfReader(pdf_path)
        page_offset = current_page
        
        # Add pages with margin text
        for page in pdf.pages:
            modified_page = add_margin_text(page, pdf_basename)
            writer.add_page(modified_page)
            
        if pdf.outline:
            seen_bookmarks = set()
            for item in pdf.outline:
                if isinstance(item, dict) and '/Page' in item:
                    title = item['/Title']
                    if title not in seen_bookmarks:
                        seen_bookmarks.add(title)
                        page_num = pdf.get_destination_page_number(item)
                        writer.add_outline_item(
                            title,
                            page_offset + page_num,
                            parent=separator_bookmark
                        )
        
        current_page += len(pdf.pages)
    
    with open(output_path, 'wb') as output:
        writer.write(output)

def verify_bookmarks(pdf_path):
    reader = PdfReader(pdf_path)
    
    def print_bookmark_tree(bookmarks, level=0):
        for item in bookmarks:
            if isinstance(item, list):
                print_bookmark_tree(item, level + 1)
            else:
                indent = "  " * level
                page_num = reader.get_destination_page_number(item)
                print(f"{indent}- {item.title} (Page {page_num})")
    
    print("\nBookmark structure:")
    print_bookmark_tree(reader.outline)

def run_tests():
    print("Creating sample PDFs...")
    
    # Create structured PDFs
    create_sample_pdf("article1.pdf", "Article 1", [
        ("Section 1.1", 0),
        ("Section 1.2", 0)
    ])
    
    create_sample_pdf("article2.pdf", "Article 2", [
        ("Section 2.1", 0),
        ("Section 2.2", 0)
    ])
    
    # Create an unstructured PDF
    create_unstructured_pdf("article3.pdf", "Article 3 - No Structure")
    
    print("Merging PDFs...")
    pdf_files = ["article1.pdf", "article2.pdf", "article3.pdf"]
    merge_pdfs_with_structure(pdf_files, "merged_articles.pdf")
    
    print("Verifying merged PDF structure...")
    verify_bookmarks("merged_articles.pdf")
    
    print("\nTest completed. Please check merged_articles.pdf")

if __name__ == "__main__":
    run_tests()


Creating sample PDFs...
Merging PDFs...
Verifying merged PDF structure...

Bookmark structure:
- Article 1: article1 (Page 1)
  - Section 1.1 (Page 2)
  - Section 1.2 (Page 2)
- Article 2: article2 (Page 3)
  - Section 2.1 (Page 4)
  - Section 2.2 (Page 4)
- Article 3: article3 (Page 5)

Test completed. Please check merged_articles.pdf
